# Task & Schedule Manager

A lightweight personal scheduler for planning tasks — reading, futsal, tournaments, or
anything else — on any date, with a quick view of what's coming up this week.

Tasks persist to a local `tasks.json` file next to this notebook, so they survive between
restarts. No database or server needed.

**Covered here:**
1. Data model (`Task`)
2. Date helpers (`"today"`, `"tomorrow"`, `"saturday"`, `"next sunday"`, or `"YYYY-MM-DD"`)
3. The `Scheduler` (add / remove / complete / query tasks)
4. Adding example tasks
5. A this-week-at-a-glance view
6. Looking up a specific date
7. Next steps (hooking this into the voice assistant)

## 1. Install dependencies

Only `pandas` is needed, for a clean table view of tasks. Everything else
(`json`, `dataclasses`, `datetime`, `uuid`, `pathlib`) is in the standard library.

In [ ]:
%pip install -q pandas

## 2. Data model

Each task has a title, a category, a date, and optional time/notes. `id` is generated
automatically so tasks can be marked done or removed later.

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, field, asdict
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Optional
import uuid

TASKS_FILE = Path("tasks.json")

CATEGORIES = ["reading", "sports", "tournament", "personal", "other"]


@dataclass
class Task:
    title: str
    date: str  # ISO format: YYYY-MM-DD
    category: str = "other"
    time: Optional[str] = None  # e.g. "18:30"
    notes: Optional[str] = None
    done: bool = False
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

    def to_dict(self) -> dict:
        return asdict(self)

## 3. Date helpers

Accepts an explicit `"YYYY-MM-DD"` string, or natural phrases: `"today"`, `"tomorrow"`,
a weekday name (`"saturday"`), or `"next <weekday>"` (`"next monday"`). This also makes it
easy to plug in transcribed speech later — "schedule futsal next saturday" already gives
you a phrase this parser understands.

In [ ]:
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]


def parse_date(text: str, today: Optional[date] = None) -> date:
    today = today or date.today()
    text = text.strip().lower()

    if text == "today":
        return today
    if text == "tomorrow":
        return today + timedelta(days=1)

    is_next = text.startswith("next ")
    weekday_name = text[5:] if is_next else text

    if weekday_name in WEEKDAYS:
        target_weekday = WEEKDAYS.index(weekday_name)
        days_ahead = (target_weekday - today.weekday()) % 7
        if is_next:
            days_ahead = days_ahead + 7 if days_ahead > 0 else 7
        return today + timedelta(days=days_ahead)

    return date.fromisoformat(text)  # falls back to "YYYY-MM-DD", raises if invalid

## 4. The scheduler

Add, remove, complete, and query tasks. Everything is saved to `tasks.json` on every
change, so you don't need to remember to call a `save()` method.

In [ ]:
class Scheduler:
    def __init__(self, storage_path: Path = TASKS_FILE):
        self.storage_path = storage_path
        self.tasks: list[Task] = self._load()

    def _load(self) -> list[Task]:
        if not self.storage_path.exists():
            return []
        raw = json.loads(self.storage_path.read_text())
        return [Task(**item) for item in raw]

    def _save(self) -> None:
        self.storage_path.write_text(json.dumps([t.to_dict() for t in self.tasks], indent=2))

    def add_task(
        self,
        title: str,
        when: str,
        category: str = "other",
        time: Optional[str] = None,
        notes: Optional[str] = None,
    ) -> Task:
        task_date = parse_date(when)
        task = Task(title=title, date=task_date.isoformat(), category=category, time=time, notes=notes)
        self.tasks.append(task)
        self._save()
        return task

    def remove_task(self, task_id: str) -> bool:
        before = len(self.tasks)
        self.tasks = [t for t in self.tasks if t.id != task_id]
        self._save()
        return len(self.tasks) < before

    def mark_done(self, task_id: str, done: bool = True) -> bool:
        for t in self.tasks:
            if t.id == task_id:
                t.done = done
                self._save()
                return True
        return False

    def tasks_on(self, when: str) -> list[Task]:
        target = parse_date(when)
        return sorted(
            (t for t in self.tasks if t.date == target.isoformat()),
            key=lambda t: t.time or "",
        )

    def upcoming(self, days: int = 7, start: Optional[date] = None) -> dict[str, list[Task]]:
        start = start or date.today()
        return {
            (start + timedelta(days=i)).isoformat(): self.tasks_on((start + timedelta(days=i)).isoformat())
            for i in range(days)
        }


scheduler = Scheduler()

## 5. Add some example tasks

Swap these for your own — same three calls handle reading, sports, and tournaments.

In [ ]:
scheduler.add_task("Finish reading 'Atomic Habits'", when="tomorrow", category="reading")
scheduler.add_task("Futsal with friends", when="saturday", category="sports", time="18:00")
scheduler.add_task(
    "Inter-college futsal tournament",
    when="next sunday",
    category="tournament",
    time="09:00",
    notes="Bring jersey + water bottle",
)

import pandas as pd

def to_dataframe(tasks: list[Task]) -> pd.DataFrame:
    columns = ["date", "time", "title", "category", "done", "id"]
    if not tasks:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame([t.to_dict() for t in tasks])[columns].sort_values(["date", "time"])

to_dataframe(scheduler.tasks)

## 6. This week at a glance

In [ ]:
for day, tasks in scheduler.upcoming(days=7).items():
    label = datetime.fromisoformat(day).strftime("%A, %d %b")
    print(f"\n{label}")
    if not tasks:
        print("  (nothing scheduled)")
    for t in tasks:
        time_str = f"{t.time} - " if t.time else ""
        box = "x" if t.done else " "
        print(f"  [{box}] {time_str}{t.title} ({t.category})")

## 7. Look up a specific date

In [ ]:
to_dataframe(scheduler.tasks_on("next sunday"))

## 8. Mark a task done, or remove it

Uncomment and pass a real task `id` (visible in the tables above).

In [ ]:
# scheduler.mark_done("a1b2c3d4")
# scheduler.remove_task("a1b2c3d4")

## Next steps

- **Voice command integration**: in `app.py`, add a `"schedule"` branch to `classify_intent`
  that extracts `title`, `when`, and `category` from the spoken request (e.g. "remind me to
  play futsal next Saturday at 6"), then calls `scheduler.add_task(...)`. A "what's on my
  schedule" style question would call `scheduler.upcoming()` or `scheduler.tasks_on(...)`
  and read the result back through `text_to_speech`.
- **Reminders**: a small cell that checks `scheduler.tasks_on("today")` against the current
  time and fires a local notification (`osascript` on macOS, `notify-send` on Linux)
  shortly before each task's `time`.
- **Bigger task list**: swap the JSON file for SQLite if this grows past a few hundred tasks.